# Deep Exploration of arXiv Paper Metadata

A thorough analysis of 1M randomly sampled papers from the arXiv repository using **Polars** (fast columnar processing) and **Plotly** (interactive visualizations).

## Setup & Data Loading

In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from collections import Counter

DATA = "arxiv_random_sample.parquet"

# Use lazy API for efficient scanning
lazy = pl.scan_parquet(DATA)

# Collect full data once; 1M rows x 14 cols fits comfortably in memory
df = lazy.collect(engine="streaming")
print(f"Shape: {df.shape}")

## 1. Schema & Basic Statistics

In [ ]:
nulls_row = df.null_count()
print("=== SCHEMA ===")
print(df.schema)
print()
print("=== NULL COUNTS ===")
for c in df.columns:
    cnt = nulls_row[c][0]
    pct = cnt / len(df) * 100
    print(f"  {c:20s}  {cnt:>8,d}  ({pct:5.1f}%)")


In [ ]:
null_df = (
    df.null_count()
    .unpivot(on=df.columns, index=[], variable_name="column", value_name="null_count")
    .with_columns(
        (pl.col("null_count") / df.height * 100).alias("null_pct"),
        (1 - pl.col("null_count") / df.height).alias("filled_pct"),
    )
)


## 2. Temporal Analysis

The `update_date` field records the most recent version update. This gives us a picture of arXiv submission and revision activity over time.

In [ ]:
date_df = (
    df.lazy()
    .with_columns(pl.col("update_date").str.to_date("%Y-%m-%d").alias("date"))
    .with_columns(
        pl.col("date").dt.year().alias("year"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.ordinal_day().alias("day_of_year"),
    )
    .collect()
)

print(f"Date range: {date_df['date'].min()}  →  {date_df['date'].max()}")

In [ ]:
year_counts = date_df.group_by("year").agg(pl.len().alias("count")).sort("year")

fig = px.bar(
    year_counts,
    x="year",
    y="count",
    title="Papers per Year (by most recent update date)",
    labels={"year": "", "count": "Papers"},
    height=450,
)
fig.update_traces(marker_color="#636efa")
fig.show()

In [ ]:
month_counts = date_df.group_by("month").agg(pl.len().alias("count")).sort("month")
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig = px.bar(
    month_counts.with_columns(pl.col("month").replace_strict(pl.Series(range(1,13)), pl.Series(month_names)).alias("month_name")),
    x="month_name",
    y="count",
    title="Papers per Month (seasonal pattern)",
    labels={"month_name": "", "count": "Papers"},
    height=400,
)
fig.update_traces(marker_color="#00cc96")
fig.show()

In [ ]:
heatmap_data = (
    date_df.filter(pl.col("year") >= 2010)
    .group_by(["year", "month"])
    .agg(pl.len().alias("count"))
    .sort(["year", "month"])
)

pivot = heatmap_data.pivot(values="count", index="year", on="month", aggregate_function="first")
months_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

month_map = {str(m): months_labels[m-1] for m in range(1, 13)}
z = pivot.select([pl.col(k).alias(v) for k, v in month_map.items()]).to_numpy()

fig = px.imshow(
    z,
    x=months_labels,
    y=pivot["year"].to_list(),
    title="Submission Activity Heatmap (Year × Month)",
    labels={"x": "Month", "y": "Year", "color": "Papers"},
    color_continuous_scale="blues",
    aspect="auto",
    height=600,
)
fig.show()


## 3. Version History

In [ ]:
version_stats = df.select(
    pl.col("versions").list.len().alias("n_versions")
)

print(version_stats.describe())
print()

# Cap at 20 for visualization
version_binned = (
    version_stats
    .with_columns(pl.when(pl.col("n_versions") > 20).then(21).otherwise(pl.col("n_versions")).alias("n_vers_binned"))
    .group_by("n_vers_binned")
    .agg(pl.len().alias("count"))
    .sort("n_vers_binned")
)

labels = [str(i) for i in range(1, 21)] + ["21+"]

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=("Versions per Paper (capped at 20)", "Zoom: 5+ versions"),
)

fig.add_trace(
    go.Bar(x=labels, y=version_binned["count"], marker_color="#636efa", showlegend=False),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=labels[4:], y=version_binned["count"][4:], marker_color="#ef553b", showlegend=False),
    row=1, col=2
)
fig.update_layout(title_text="Distribution of Number of Versions", height=450)
fig.show()

## 4. Category Analysis

Papers can have multiple categories (primary + cross-list). We split on spaces and count individual tags first, then examine co-occurrence patterns.

In [ ]:
# Count categories per paper
cat_counts = df.select(
    pl.col("categories").str.split(" ").list.len().alias("n_cats")
)

print("Categories per paper:")
print(cat_counts.describe())
print()

fig = px.histogram(
    cat_counts.to_pandas().clip(upper=10),
    x="n_cats",
    nbins=10,
    title="Number of Categories per Paper",
    labels={"n_cats": "Categories", "count": "Papers"},
    height=400,
)
fig.update_traces(marker_color="#ab63fa")
fig.update_xaxes(tickvals=list(range(1, 11)))
fig.show()

In [ ]:
# Split categories and count individual tags
all_cats = (
    df.lazy()
    .select(pl.col("categories").str.split(" "))
    .with_columns(pl.col("categories").list.eval(pl.element().alias("cat")).alias("exploded"))
    .select(pl.col("exploded"))
    .collect()
)

# Polars explode
flat_cats = df.select(pl.col("categories").str.split(" ")).explode("categories")
top_cats = flat_cats.group_by("categories").agg(pl.len().alias("count")).sort("count", descending=True).head(30)

fig = px.bar(
    top_cats,
    x="count",
    y="categories",
    orientation="h",
    title="Top 30 Individual arXiv Categories",
    labels={"count": "Papers", "categories": ""},
    height=700,
    text_auto=True,
)
fig.update_traces(marker_color="#00cc96")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
# Co-occurrence matrix for top-10 categories
top10 = [c for c in top_cats["categories"][:10]]

cat_matrix = df.select(
    *[pl.col("categories").str.contains(c, literal=True).cast(pl.Int32).alias(c) for c in top10]
)

# Build co-occurrence counts
cooc = np.zeros((10, 10), dtype=int)
for i, c1 in enumerate(top10):
    for j, c2 in enumerate(top10):
        cooc[i, j] = cat_matrix.filter(pl.col(c1) == 1).filter(pl.col(c2) == 1).height

# Normalize to co-occur / total(c1)
totals = np.diag(cooc).copy()
cooc_norm = cooc / totals[np.newaxis, :].astype(float)
np.fill_diagonal(cooc_norm, np.nan)

fig = px.imshow(
    cooc_norm,
    x=[c.split(".")[-1] for c in top10],
    y=[c.split(".")[-1] for c in top10],
    title="Category Co-occurrence (fraction of row cat that also have col cat)",
    labels={"x": "Co-occurring category", "y": "Category", "color": "Fraction"},
    color_continuous_scale="blues",
    text_auto=".0%",
    height=600,
    width=700,
)
fig.show()

## 5. License Distribution

In [ ]:
lic_counts = df.group_by("license").agg(pl.len().alias("count")).sort("count", descending=True)

short_labels = (
    lic_counts.lazy()
    .with_columns(
        pl.when(pl.col("license").str.contains("nonexclusive", literal=True))
        .then(pl.lit("arXiv nonexclusive"))
        .when(pl.col("license").str.contains("by/4.0", literal=True))
        .then(pl.lit("CC BY 4.0"))
        .when(pl.col("license").str.contains("by-nc-nd/4.0", literal=True))
        .then(pl.lit("CC BY-NC-ND 4.0"))
        .when(pl.col("license").str.contains("by-nc-sa/4.0", literal=True))
        .then(pl.lit("CC BY-NC-SA 4.0"))
        .when(pl.col("license").str.contains("by-sa/4.0", literal=True))
        .then(pl.lit("CC BY-SA 4.0"))
        .when(pl.col("license").str.contains("publicdomain/zero", literal=True))
        .then(pl.lit("CC0 1.0"))
        .when(pl.col("license").is_null())
        .then(pl.lit("Missing"))
        .otherwise(pl.lit("Other"))
        .alias("license_short")
    )
    .collect()
)

fig = px.pie(
    short_labels,
    values="count",
    names="license_short",
    title="License Distribution",
    hole=0.4,
    height=500,
)
fig.update_traces(textinfo="label+percent")
fig.show()

## 6. Author Analysis

Using the parsed `authors_parsed` field (list of `[last, first, suffix]` arrays) and the plain-text `authors` field.

In [ ]:
n_authors = df.select(
    pl.col("authors_parsed").list.len().alias("n_authors")
)

print("Authors per paper:")
print(n_authors.describe())
print(f"  Papers with > 100 authors: {n_authors.filter(pl.col('n_authors') > 100).height:,}")
print(f"  Papers with > 1000 authors: {n_authors.filter(pl.col('n_authors') > 1000).height:,}")
print(f"  Max authors: {n_authors['n_authors'].max():,}")

In [ ]:
binned = (
    n_authors
    .with_columns(
        pl.when(pl.col("n_authors") > 50).then(51)
        .otherwise(pl.col("n_authors"))
        .alias("n_auth_binned")
    )
    .group_by("n_auth_binned")
    .agg(pl.len().alias("count"))
    .sort("n_auth_binned")
)

labels = [str(i) for i in range(1, 51)] + ["51+"]

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=("Authors per Paper (capped at 50)", "Zoom: 11+ authors"),
)

fig.add_trace(
    go.Bar(x=labels, y=binned["count"], marker_color="#636efa", showlegend=False),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=labels[10:], y=binned["count"][10:], marker_color="#ef553b", showlegend=False),
    row=1, col=2
)
fig.update_layout(title_text="Distribution of Number of Authors", height=450)
fig.show()

In [ ]:
# Extract last names from authors_parsed
last_names = (
    df.lazy()
    .select(pl.col("authors_parsed").list.eval(pl.element().list.first().str.to_titlecase()))
    .collect()
)

# Explode and count
author_counts = (
    last_names
    .explode("authors_parsed")
    .group_by("authors_parsed")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(20)
)

fig = px.bar(
    author_counts,
    x="count",
    y="authors_parsed",
    orientation="h",
    title="Top 20 Author Last Names in Sample",
    labels={"count": "Papers", "authors_parsed": ""},
    height=600,
    text_auto=True,
)
fig.update_traces(marker_color="#ffa15a")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 7. Text Length Analysis

Analyzing the lengths of titles and abstracts to understand the textual characteristics of arXiv papers.

In [ ]:
text_df = df.select(
    pl.col("title").str.len_chars().alias("title_len"),
    pl.col("abstract").str.len_chars().alias("abstract_len"),
)

print("=== TITLE LENGTH (chars) ===")
print(text_df["title_len"].describe())
print()
print("=== ABSTRACT LENGTH (chars) ===")
print(text_df["abstract_len"].describe())
print()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Title Length", "Abstract Length"),
    shared_yaxes=True,
)

fig.add_trace(go.Histogram(x=text_df["title_len"], nbinsx=60, marker_color="#636efa", name="Title"), row=1, col=1)
fig.add_trace(go.Histogram(x=text_df["abstract_len"].to_pandas().clip(upper=3000), nbinsx=80, marker_color="#00cc96", name="Abstract"), row=1, col=2)
fig.update_layout(title_text="Text Length Distributions", height=450, showlegend=False)
fig.update_xaxes(title_text="Characters", row=1, col=1)
fig.update_xaxes(title_text="Characters (capped at 3000)", row=1, col=2)
fig.show()

In [ ]:
# Scatter: title length vs abstract length (sample 10k for speed)
rng = np.random.default_rng(42)
idx = rng.choice(len(text_df), size=10_000, replace=False)
sample_text = text_df.to_pandas().iloc[idx]

fig = px.scatter(
    sample_text,
    x="title_len",
    y="abstract_len",
    title="Title Length vs Abstract Length (10k sample)",
    labels={"title_len": "Title length (chars)", "abstract_len": "Abstract length (chars)"},
    opacity=0.3,
    height=500,
)
fig.show()

In [ ]:
wc = df.select(
    pl.col("abstract").str.split(" ").list.len().alias("abstract_words")
)


## 8. Comments Field Analysis

The `comments` field often contains structured metadata like page counts, figure counts, or conference info.

In [ ]:
comments_df = df.select(
    pl.col("comments").str.len_chars().alias("comments_len"),
    pl.col("comments").str.contains(r"\d+\s*pages?", literal=False).alias("has_page_count"),
    pl.col("comments").str.contains(r"\d+\s*figures?", literal=False).alias("has_figure_count"),
    pl.col("comments").str.contains(r"\d+\s*\w*\s*references?", literal=False).alias("has_ref_count"),
)

print(f"Papers with page count in comments: {comments_df['has_page_count'].sum():>8,d}  ({comments_df['has_page_count'].mean()*100:.1f}%)")
print(f"Papers with figure count:         {comments_df['has_figure_count'].sum():>8,d}  ({comments_df['has_figure_count'].mean()*100:.1f}%)")
print(f"Papers with reference count:      {comments_df['has_ref_count'].sum():>8,d}  ({comments_df['has_ref_count'].mean()*100:.1f}%)")
print()
print("Comments length stats:")
print(comments_df["comments_len"].describe())

In [ ]:
# Show a few interesting comment examples
non_null_comments = df.filter(pl.col("comments").is_not_null()).select("comments").unique()
print("=== SAMPLE COMMENTS ===")
for c in non_null_comments["comments"][:15]:
    print(f"  • {c[:120]}")

## 9. arXiv ID Analysis

arXiv IDs follow patterns like `YYMM.NNNNN` (post-2007) or `arch-ive/YYMMNNN` (older format). Let's check the breakdown.

In [ ]:
id_df = df.select(
    pl.col("id").str.contains("/", literal=True).alias("is_old_format"),
    pl.col("id").str.len_chars().alias("id_len"),
)

old_count = id_df["is_old_format"].sum()
new_count = id_df.height - old_count
print(f"Old format (with /): {old_count:>10,d}  ({old_count/id_df.height*100:.2f}%)")
print(f"New format (no /):  {new_count:>10,d}  ({new_count/id_df.height*100:.2f}%)")
print()
print("ID length distribution:")
id_len_counts = id_df.group_by("id_len").agg(pl.len().alias("count")).sort("id_len")
print(id_len_counts)

fig = px.bar(
    id_len_counts.sort("id_len"),
    x="id_len",
    y="count",
    title="arXiv ID Length Distribution",
    labels={"id_len": "Characters", "count": "Papers"},
    height=400,
)
fig.update_traces(marker_color="#636efa")
fig.show()

## 10. Missing Data Patterns

Examining how null values correlate across columns.

In [ ]:
# Create indicator matrix for nulls
null_cols = [c for c in df.columns if df[c].null_count() > 0]
null_indicators = df.select(
    *[pl.col(c).is_null().alias(c) for c in null_cols]
)

# Co-null counts
print("=== PAIRWISE CO-NULL COUNTS ===")
null_mat = null_indicators.to_numpy()
comat = null_mat.T @ null_mat
for i, c1 in enumerate(null_cols):
    for j, c2 in enumerate(null_cols):
        if i < j:
            print(f"  {c1:15s} & {c2:15s}:  {comat[i,j]:>8,d}")

In [ ]:
# How many papers have NO metadata beyond id/title/abstract?
meta_cols = ["submitter", "comments", "journal-ref", "doi", "report-no", "license"]
n_missing_meta = (
    df.select(pl.sum_horizontal(pl.col(c).is_null().cast(pl.Int32) for c in meta_cols).alias("missing_meta"))
)

missing_dist = n_missing_meta.group_by("missing_meta").agg(pl.len().alias("count")).sort("missing_meta")

fig = px.bar(
    missing_dist,
    x="missing_meta",
    y="count",
    title="Number of Missing Metadata Fields (out of 6)",
    labels={"missing_meta": "Missing fields", "count": "Papers"},
    height=400,
    text_auto=True,
)
fig.update_traces(marker_color="#ef553b")
fig.show()

## Summary of Key Findings

- **Size**: 1,000,000 papers, 14 columns, ~1.6 GB in memory
- **Temporal range**: Papers span decades; heaviest activity in recent years with a strong September/October peak (start of academic year)
- **Versions**: ~55% have only one version; mean is 1.54; a handful have 100+ revisions
- **Categories**: `astro-ph`, `hep-ph`, `quant-ph`, `hep-th`, and `cs.CV` dominate. Most papers have 1-2 categories
- **License**: 40% arXiv nonexclusive, 33% missing, 20% CC BY 4.0
- **Authors**: Most papers have 1-5 authors; mean 4.45; some HEP papers have 3000+ (large collaborations)
- **Missing data**: `report-no` (91%), `journal-ref` (68%), `doi` (59%) are the sparsest fields
- **Text**: Titles average 75 chars, abstracts average 980 chars (~150 words)